# 09. Figury, EDA i materia?y do pracy


Notebook grupuje kod generuj?cy figury, tabele pomocnicze i grafiki teoretyczne u?ywane w pracy.
To jest g??wny zeszyt odtwarzaj?cy wizualn? warstw? pracy magisterskiej.
Kod pochodzi ze skrypt?w `09_thesis_figure_pack.py`, `14_thesis_narrative_case_study_figures.py`, `15_thesis_eda_figures.py` i `16_theory_figures.py`.

**Uwaga:** kom?rki z kodem zawieraj? ?r?d?owe skrypty, ale wywo?anie `main()` jest celowo zakomentowane. Dzi?ki temu przypadkowe `Run All` nie uruchomi d?ugich pobra? danych ani trening?w. Aby wykona? dan? sekcj?, najlepiej uruchomi? j? ?wiadomie albo u?y? `%run nazwa_skryptu.py`.


In [ ]:
from pathlib import Path

ROOT = Path.cwd()
EXPORT_DIR = ROOT / 'exports'
FIG_DIR = ROOT / 'figures' / 'thesis'

print(f'Root: {ROOT}')
print(f'Exports: {EXPORT_DIR}')
print(f'Figures: {FIG_DIR}')


## Figury EDA: rozmiary zbior?w, redukcja danych, d?ugo?ci sekwencji, korelacje

?r?d?o: `15_thesis_eda_figures.py`

Je?eli chcesz uruchomi? oryginalny skrypt bez wchodzenia w definicje funkcji, u?yj:

```python
%run 15_thesis_eda_figures.py
```


In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd


ROOT = Path(__file__).resolve().parent
EXPORTS = ROOT / "exports"
FIG_DIR = ROOT / "figures" / "thesis"
FIG_DIR.mkdir(parents=True, exist_ok=True)


SETUPS = {
    "Kwalifikacje\n2023-2025": {
        "file": EXPORTS / "lap_features_model_min_40_laps.csv",
        "drivers": ["ALB", "SAI", "VER", "OCO", "LEC"],
        "filter_file": EXPORTS / "lap_filter_summary.csv",
        "telemetry_file": EXPORTS / "best_laps_telemetry_audit.csv",
    },
    "Kwalifikacje\n2018-2025": {
        "file": EXPORTS / "balanced_top5_lap_features_2018_2025_strict.csv",
        "drivers": ["SAI", "VER", "LEC", "OCO", "HAM"],
        "filter_file": EXPORTS / "lap_filter_summary_2018_2025_strict.csv",
        "telemetry_file": EXPORTS / "balanced_top6_telemetry_audit_2018_2025_strict.csv",
    },
    "Wyścigi\n2025": {
        "file": EXPORTS / "race_2025_balanced_top5_lap_features.csv",
        "drivers": ["SAI", "VER", "LEC", "OCO", "HAM"],
        "filter_file": EXPORTS / "race_2025_filter_summary.csv",
        "telemetry_file": EXPORTS / "race_2025_balanced_top5_telemetry_audit.csv",
    },
}


STYLE_FEATURES = [
    "speed_mean",
    "speed_std",
    "speed_min",
    "speed_max",
    "throttle_mean",
    "throttle_std",
    "throttle_zero_frac",
    "throttle_full_frac",
    "brake_active_frac",
    "brake_on_count",
    "rpm_mean",
    "rpm_std",
    "gear_mean",
    "gear_change_count",
]


def savefig(name: str) -> None:
    plt.tight_layout()
    plt.savefig(FIG_DIR / name, dpi=180, bbox_inches="tight")
    plt.close()


def load_setup_frames() -> dict[str, pd.DataFrame]:
    frames = {}
    for setup_name, cfg in SETUPS.items():
        df = pd.read_csv(cfg["file"])
        if "Driver" in df.columns:
            df = df[df["Driver"].isin(cfg["drivers"])].copy()
        df["eda_setup"] = setup_name.replace("\n", " ")
        frames[setup_name] = df
    return frames


def plot_dataset_sizes(frames: dict[str, pd.DataFrame]) -> None:
    rows = []
    for setup_name, cfg in SETUPS.items():
        filter_df = pd.read_csv(cfg["filter_file"])
        raw = int(filter_df.iloc[0]["n_rows"])
        filtered_pool = int(filter_df.iloc[-1]["n_rows"])
        model_rows = len(frames[setup_name])
        rows.append(
            {
                "setup": setup_name.replace("\n", " "),
                "raw_rows": raw,
                "filtered_pool_rows": filtered_pool,
                "model_rows": model_rows,
                "retention_raw_to_model": model_rows / raw,
                "retention_filtered_to_model": model_rows / filtered_pool,
            }
        )
    overview = pd.DataFrame(rows)
    overview.to_csv(EXPORTS / "thesis_eda_dataset_overview.csv", index=False)

    x = np.arange(len(overview))
    width = 0.26
    fig, ax = plt.subplots(figsize=(10, 5.5))
    ax.bar(x - width, overview["raw_rows"], width, label="surowe okrążenia", color="#9aa7b4")
    ax.bar(x, overview["filtered_pool_rows"], width, label="po głównych filtrach", color="#4d7c9f")
    ax.bar(x + width, overview["model_rows"], width, label="w modelu", color="#c7673b")
    ax.set_yscale("log")
    ax.set_ylabel("Liczba okrążeń (skala log)")
    ax.set_title("Skala danych przed i po filtracji")
    ax.set_xticks(x)
    ax.set_xticklabels(overview["setup"], rotation=0)
    ax.legend(frameon=False)
    ax.grid(axis="y", alpha=0.25)
    savefig("24_eda_dataset_sizes.png")


def plot_class_balance(frames: dict[str, pd.DataFrame]) -> None:
    balance_rows = []
    for setup_name, df in frames.items():
        counts = df["Driver"].value_counts().sort_index()
        for driver, n in counts.items():
            balance_rows.append({"setup": setup_name.replace("\n", " "), "Driver": driver, "n_laps": n})
    balance = pd.DataFrame(balance_rows)
    balance.to_csv(EXPORTS / "thesis_eda_class_balance.csv", index=False)

    fig, axes = plt.subplots(1, 3, figsize=(13, 4), sharey=False)
    colors = ["#5b7894", "#c7673b", "#75845c"]
    for ax, (setup_name, group), color in zip(axes, balance.groupby("setup", sort=False), colors):
        group = group.sort_values("n_laps", ascending=False)
        ax.bar(group["Driver"], group["n_laps"], color=color)
        ax.set_title(setup_name)
        ax.set_ylabel("Liczba okrążeń")
        ax.grid(axis="y", alpha=0.25)
    fig.suptitle("Liczebność klas kierowców w finalnych setupach", y=1.04)
    savefig("25_eda_class_balance.png")


def feature_type_inventory(frames: dict[str, pd.DataFrame]) -> None:
    id_like = {
        "lap_key",
        "Driver",
        "Team",
        "Compound",
        "session_key",
        "event_name",
        "TrackStatus",
        "SessionType",
        "eda_setup",
    }
    rows = []
    for setup_name, df in frames.items():
        numeric_cols = df.select_dtypes(include=[np.number]).columns.tolist()
        categorical_cols = [c for c in df.columns if c not in numeric_cols]
        identifier_cols = [c for c in df.columns if c in id_like or c.lower().endswith("key")]
        model_candidate_numeric = [c for c in numeric_cols if c not in identifier_cols]
        rows.append(
            {
                "setup": setup_name.replace("\n", " "),
                "n_rows": len(df),
                "n_columns": df.shape[1],
                "numeric_columns": len(numeric_cols),
                "categorical_columns": len(categorical_cols),
                "identifier_or_context_columns": len(identifier_cols),
                "numeric_model_candidate_columns": len(model_candidate_numeric),
            }
        )
    pd.DataFrame(rows).to_csv(EXPORTS / "thesis_eda_feature_type_inventory.csv", index=False)


def plot_missingness(frames: dict[str, pd.DataFrame]) -> None:
    excluded = {
        "season",
        "round",
        "LapNumber",
        "lap_key",
        "Driver",
        "Team",
        "Compound",
        "session_key",
        "event_name",
        "SessionType",
        "SessionPart",
        "stint_id",
        "lap_time_delta_to_stint_median",
        "stint_median_lap_seconds",
        "eda_setup",
    }
    rows = []
    for setup_name, df in frames.items():
        numeric_cols = [c for c in df.select_dtypes(include=[np.number]).columns if c not in excluded]
        miss = df[numeric_cols].isna().mean().sort_values(ascending=False)
        nonzero = miss.head(10)
        for feature, frac in nonzero.items():
            rows.append({"setup": setup_name.replace("\n", " "), "feature": feature, "missing_frac": frac})
    missing = pd.DataFrame(rows)
    missing.to_csv(EXPORTS / "thesis_eda_missingness_top.csv", index=False)

    fig, axes = plt.subplots(1, 3, figsize=(14, 5), sharex=False)
    for ax, (setup_name, group) in zip(axes, missing.groupby("setup", sort=False)):
        group = group.sort_values("missing_frac", ascending=True)
        ax.barh(group["feature"], group["missing_frac"] * 100, color="#5b7894")
        ax.set_title(setup_name)
        ax.set_xlabel("Braki danych [%]")
        ax.grid(axis="x", alpha=0.25)
    fig.suptitle("Braki danych w numerycznych cechach modelowych", y=1.02)
    savefig("26_eda_missingness.png")


def plot_sequence_lengths(frames: dict[str, pd.DataFrame]) -> None:
    rows = []
    for setup_name, cfg in SETUPS.items():
        audit = pd.read_csv(cfg["telemetry_file"])
        if "Driver" in audit.columns:
            audit = audit[audit["Driver"].isin(cfg["drivers"])].copy()
        for value in audit["merged_rows"].dropna():
            rows.append({"setup": setup_name.replace("\n", " "), "merged_rows": float(value)})
    seq = pd.DataFrame(rows)
    seq.to_csv(EXPORTS / "thesis_eda_sequence_lengths.csv", index=False)

    fig, ax = plt.subplots(figsize=(9, 5))
    labels = []
    data = []
    for setup_name, group in seq.groupby("setup", sort=False):
        labels.append(setup_name)
        data.append(group["merged_rows"].to_numpy())
    ax.boxplot(data, tick_labels=labels, showfliers=False, patch_artist=True)
    ax.set_ylabel("Liczba próbek telemetrycznych w okrążeniu")
    ax.set_title("Długość sekwencji telemetrycznych przed resamplingiem")
    ax.grid(axis="y", alpha=0.25)
    savefig("27_eda_sequence_lengths.png")


def plot_feature_distributions(frames: dict[str, pd.DataFrame]) -> None:
    combined = pd.concat(frames.values(), ignore_index=True)
    features = [
        "speed_mean",
        "speed_std",
        "throttle_full_frac",
        "throttle_zero_frac",
        "brake_active_frac",
        "gear_mean",
    ]
    existing = [f for f in features if f in combined.columns]
    rows = []
    for feature in existing:
        for setup, group in combined.groupby("eda_setup", sort=False):
            values = pd.to_numeric(group[feature], errors="coerce").dropna()
            if len(values) == 0:
                continue
            rows.append(
                {
                    "feature": feature,
                    "setup": setup,
                    "mean": values.mean(),
                    "std": values.std(),
                    "p25": values.quantile(0.25),
                    "median": values.median(),
                    "p75": values.quantile(0.75),
                }
            )
    pd.DataFrame(rows).to_csv(EXPORTS / "thesis_eda_key_feature_summary.csv", index=False)

    fig, axes = plt.subplots(2, 3, figsize=(13, 7))
    axes = axes.ravel()
    for ax, feature in zip(axes, existing):
        plot_data = []
        labels = []
        for setup, group in combined.groupby("eda_setup", sort=False):
            values = pd.to_numeric(group[feature], errors="coerce").dropna()
            plot_data.append(values)
            labels.append(setup.replace(" ", "\n"))
        ax.boxplot(plot_data, tick_labels=labels, showfliers=False, patch_artist=True)
        ax.set_title(feature)
        ax.grid(axis="y", alpha=0.25)
    fig.suptitle("Rozkłady wybranych cech telemetrycznych", y=1.02)
    savefig("28_eda_feature_distributions.png")


def plot_correlation_heatmap(frames: dict[str, pd.DataFrame]) -> None:
    common_features = [f for f in STYLE_FEATURES if all(f in df.columns for df in frames.values())]
    combined = pd.concat([df[common_features].assign(setup=name) for name, df in frames.items()], ignore_index=True)
    corr = combined[common_features].corr()
    corr.to_csv(EXPORTS / "thesis_eda_selected_feature_correlation.csv")

    fig, ax = plt.subplots(figsize=(9.5, 8))
    im = ax.imshow(corr, cmap="RdBu_r", vmin=-1, vmax=1)
    ax.set_xticks(np.arange(len(common_features)))
    ax.set_yticks(np.arange(len(common_features)))
    ax.set_xticklabels(common_features, rotation=45, ha="right")
    ax.set_yticklabels(common_features)
    ax.set_title("Korelacje wybranych cech telemetrycznych")
    cbar = fig.colorbar(im, ax=ax, fraction=0.046, pad=0.04)
    cbar.set_label("Korelacja Pearsona")
    savefig("29_eda_feature_correlation_heatmap.png")


def main() -> None:
    frames = load_setup_frames()
    plot_dataset_sizes(frames)
    plot_class_balance(frames)
    feature_type_inventory(frames)
    plot_missingness(frames)
    plot_sequence_lengths(frames)
    plot_feature_distributions(frames)
    plot_correlation_heatmap(frames)
    print("Generated EDA figures 24-29 and CSV summaries.")


# Notebook safety: odkomentuj, je?eli chcesz uruchomi? t? sekcj?.
# main()


## Figury teoretyczne: sigmoid, CNN 1D, walidacja grupowana

?r?d?o: `16_theory_figures.py`

Je?eli chcesz uruchomi? oryginalny skrypt bez wchodzenia w definicje funkcji, u?yj:

```python
%run 16_theory_figures.py
```


In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np


ROOT = Path(__file__).resolve().parent
FIG_DIR = ROOT / "figures" / "thesis"
FIG_DIR.mkdir(parents=True, exist_ok=True)


def savefig(name: str) -> None:
    plt.tight_layout()
    plt.savefig(FIG_DIR / name, dpi=180, bbox_inches="tight")
    plt.close()


def logistic_sigmoid() -> None:
    x = np.linspace(-8, 8, 400)
    y = 1 / (1 + np.exp(-x))

    fig, ax = plt.subplots(figsize=(7.5, 4.5))
    ax.plot(x, y, color="#2f6f8f", linewidth=3)
    ax.axhline(0.5, color="#444444", linestyle="--", linewidth=1.3)
    ax.axvline(0, color="#c7673b", linestyle="--", linewidth=1.3)
    ax.fill_between(x, 0, y, where=x < 0, color="#9fb7c9", alpha=0.25, label="niższe prawdopodobieństwo klasy")
    ax.fill_between(x, 0, y, where=x >= 0, color="#e0a37d", alpha=0.25, label="wyższe prawdopodobieństwo klasy")
    ax.set_title("Funkcja logistyczna jako przejście od wyniku liniowego do prawdopodobieństwa")
    ax.set_xlabel("wynik liniowy modelu")
    ax.set_ylabel("prawdopodobieństwo klasy")
    ax.set_ylim(-0.03, 1.03)
    ax.grid(alpha=0.25)
    ax.legend(frameon=False, loc="lower right")
    savefig("30_theory_logistic_sigmoid.png")


def cnn_1d_concept() -> None:
    fig, ax = plt.subplots(figsize=(10, 4.8))
    ax.axis("off")

    colors = {
        "input": "#d9e6ef",
        "conv": "#f0c38e",
        "pool": "#d7d4f0",
        "dense": "#b9d8b4",
        "out": "#e5a1a1",
    }

    blocks = [
        (0.05, 0.32, 0.16, 0.36, "Sekwencja\ntelemetrii\nSpeed, Throttle,\nBrake, RPM, nGear", colors["input"]),
        (0.29, 0.38, 0.13, 0.24, "Filtry\nkonwolucyjne\n1D", colors["conv"]),
        (0.49, 0.38, 0.12, 0.24, "Agregacja\nlokalnych\nwzorców", colors["pool"]),
        (0.68, 0.38, 0.12, 0.24, "Warstwa\nłącząca", colors["dense"]),
        (0.86, 0.38, 0.09, 0.24, "Kierowca", colors["out"]),
    ]

    for x, y, w, h, text, color in blocks:
        rect = plt.Rectangle((x, y), w, h, facecolor=color, edgecolor="#333333", linewidth=1.5)
        ax.add_patch(rect)
        ax.text(x + w / 2, y + h / 2, text, ha="center", va="center", fontsize=10)

    for i in range(len(blocks) - 1):
        x1 = blocks[i][0] + blocks[i][2]
        y1 = blocks[i][1] + blocks[i][3] / 2
        x2 = blocks[i + 1][0]
        y2 = blocks[i + 1][1] + blocks[i + 1][3] / 2
        ax.annotate("", xy=(x2 - 0.015, y2), xytext=(x1 + 0.015, y1), arrowprops=dict(arrowstyle="->", linewidth=1.8))

    t = np.linspace(0, 1, 120)
    for i, offset in enumerate([0.78, 0.73, 0.68, 0.63, 0.58]):
        ax.plot(0.06 + 0.13 * t, offset + 0.025 * np.sin(2 * np.pi * (t * (i + 1.2))), color="#2f6f8f", linewidth=1.2)

    ax.text(0.5, 0.12, "CNN 1D szuka lokalnych wzorców w przebiegu okrążenia, np. fragmentów hamowania, wejścia na gaz albo zmian biegów.", ha="center", fontsize=11)
    savefig("31_theory_1d_cnn_concept.png")


def grouped_cv_concept() -> None:
    fig, ax = plt.subplots(figsize=(10, 4.8))
    ax.set_title("Walidacja grupowana: okrążenia z tej samej sesji nie trafiają jednocześnie do treningu i testu")
    ax.set_xlim(0, 10)
    ax.set_ylim(0, 5)
    ax.axis("off")

    sessions = ["Sesja A", "Sesja B", "Sesja C", "Sesja D", "Sesja E"]
    colors = ["#4c78a8", "#59a14f", "#f28e2b", "#b07aa1", "#e15759"]
    for i, (session, color) in enumerate(zip(sessions, colors)):
        x = 0.7 + i * 1.8
        for j in range(4):
            ax.add_patch(plt.Rectangle((x + j * 0.22, 3.3), 0.18, 0.35, facecolor=color, edgecolor="white"))
        ax.text(x + 0.35, 3.85, session, ha="center", fontsize=10)

    folds = [
        ("Fold 1", ["train", "train", "test", "train", "train"]),
        ("Fold 2", ["test", "train", "train", "train", "train"]),
        ("Fold 3", ["train", "test", "train", "train", "test"]),
    ]
    for row, (fold, states) in enumerate(folds):
        y = 2.35 - row * 0.7
        ax.text(0.05, y + 0.13, fold, ha="left", fontsize=10)
        for i, state in enumerate(states):
            x = 0.75 + i * 1.8
            face = "#d9e6ef" if state == "train" else "#c7673b"
            label = "trening" if state == "train" else "test"
            ax.add_patch(plt.Rectangle((x, y), 1.05, 0.38, facecolor=face, edgecolor="#333333", linewidth=1))
            ax.text(x + 0.525, y + 0.19, label, ha="center", va="center", fontsize=9)

    ax.text(5, 0.25, "Taki podział zmniejsza ryzyko, że model nauczy się specyfiki jednej sesji zamiast cech kierowcy.", ha="center", fontsize=11)
    savefig("32_theory_grouped_cv.png")


def main() -> None:
    logistic_sigmoid()
    cnn_1d_concept()
    grouped_cv_concept()
    print("Generated theory figures 30-32.")


# Notebook safety: odkomentuj, je?eli chcesz uruchomi? t? sekcj?.
# main()


## G??wne figury wynikowe do rozdzia??w badawczych

?r?d?o: `09_thesis_figure_pack.py`

Je?eli chcesz uruchomi? oryginalny skrypt bez wchodzenia w definicje funkcji, u?yj:

```python
%run 09_thesis_figure_pack.py
```


In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd
import seaborn as sns


EXPORT_DIR = Path("exports")
FIG_DIR = Path("figures") / "thesis"
FIG_DIR.mkdir(parents=True, exist_ok=True)

sns.set_theme(style="whitegrid", context="talk")
plt.rcParams["figure.dpi"] = 150
plt.rcParams["savefig.dpi"] = 220
plt.rcParams["font.family"] = "DejaVu Sans"


def load_csv(name: str) -> pd.DataFrame:
    return pd.read_csv(EXPORT_DIR / f"{name}.csv")


def savefig(name: str) -> Path:
    path = FIG_DIR / name
    plt.tight_layout()
    plt.savefig(path, bbox_inches="tight")
    plt.close()
    return path


def model_label(model: str) -> str:
    return {
        "cnn": "CNN",
        "hybrid_cnn_tabular": "Hybryda",
        "gru": "GRU",
        "lstm": "LSTM",
    }.get(model, model)


def pretty_dataset(dataset: str) -> str:
    return {
        "short_qualifying_2023_2025": "Kwalifikacje\n2023-2025",
        "short_qualifying_2023_2025_top5": "Kwalifikacje\n2023-2025",
        "long_qualifying_2018_2025_top5": "Kwalifikacje\n2018-2025",
        "race_2025_clean_laps_top5": "Wyścigi\n2025",
    }.get(dataset, dataset)


def plot_filter_funnels() -> list[Path]:
    setup_specs = [
        ("Kwalifikacje 2023-2025", load_csv("lap_filter_summary"), "01a_filter_funnel_qualifying_2023_2025.png"),
        ("Kwalifikacje 2018-2025", load_csv("lap_filter_summary_2018_2025_strict"), "01b_filter_funnel_qualifying_2018_2025.png"),
        ("Wyścigi 2025", load_csv("race_2025_filter_summary"), "01c_filter_funnel_race_2025.png"),
    ]
    label_map = {
        "00_raw_all_qualifying_laps": "Surowe\ndane",
        "00_raw_race_laps": "Surowe\nokrążenia",
        "01_lap_time_not_null": "Poprawny\nczas",
        "02_dry_sessions_only": "Suche\nsesje",
        "03_is_accurate_true": "Dokładne\nokrążenia",
        "04_not_deleted": "Nieusunięte",
        "05_not_fastf1_generated": "Niegenerowane",
        "06_track_status_eq_1": "Zielony\ntor",
        "07a_personal_best_only": "Najlepsze\nokrążenie",
        "07_not_pit_in_out": "Bez\npit-stopu",
        "08_within_2s_of_stint_median": "Blisko mediany\nstintu",
    }
    paths = []
    for setup, df, filename in setup_specs:
        plot_df = df.copy()
        start = plot_df["n_rows"].iloc[0]
        plot_df["retention_pct"] = plot_df["n_rows"] / start * 100
        plot_df["label"] = plot_df["step"].map(label_map).fillna(plot_df["step"])

        fig, ax = plt.subplots(figsize=(9.2, 4.9))
        sns.lineplot(data=plot_df, x="label", y="retention_pct", marker="o", linewidth=2.5, ax=ax, color="#4c78a8")
        ax.set_title(setup, fontsize=16, pad=10)
        ax.set_xlabel("Etapy filtrowania", fontsize=10, labelpad=6)
        ax.set_ylabel("Pozostało [%]", fontsize=12)
        ax.set_ylim(0, 105)
        ax.grid(axis="y", alpha=0.25)
        ax.tick_params(axis="x", rotation=0, labelsize=9)
        ax.tick_params(axis="y", labelsize=10)
        for x_pos, row in enumerate(plot_df.itertuples(index=False)):
            if x_pos in {0, len(plot_df) - 1}:
                ax.annotate(
                    f"{row.retention_pct:.1f}%",
                    (x_pos, row.retention_pct),
                    textcoords="offset points",
                    xytext=(0, 8),
                    ha="center",
                    fontsize=10,
                    color="#2f4b7c",
                )
        paths.append(savefig(filename))
    return paths


def plot_model_progression() -> Path:
    rows = []

    short = load_csv("final_results_model_comparison")
    for _, row in short.iterrows():
        label = {
            "handcrafted": "Regresja logistyczna",
            "sequence": "CNN",
            "hybrid": "Hybryda",
        }.get(row["model_family"], row["model"])
        rows.append({"setup": "Kwalifikacje\n2023-2025", "model": label, "macro_f1": row["oof_macro_f1"]})

    long_seq = load_csv("long_horizon_sequence_model_summary")
    long_base = load_csv("long_horizon_baseline_model_summary")
    long_log = long_base[
        (long_base["subset_name"] == "balanced_top5")
        & (long_base["experiment"] == "with_time_features")
        & (long_base["model"] == "logistic_regression")
    ]["oof_macro_f1"].iloc[0]
    rows.append({"setup": "Kwalifikacje\n2018-2025", "model": "Regresja logistyczna", "macro_f1": long_log})
    for model in ["cnn", "hybrid_cnn_tabular", "gru", "lstm"]:
        val = long_seq[(long_seq["subset_name"] == "balanced_top5") & (long_seq["model"] == model)]["oof_macro_f1"].iloc[0]
        rows.append({"setup": "Kwalifikacje\n2018-2025", "model": model_label(model), "macro_f1": val})

    race_seq = load_csv("race_2025_sequence_model_summary")
    race_base = load_csv("race_2025_baseline_model_summary")
    race_log = race_base[
        (race_base["experiment"] == "style_core_no_time_no_race_context")
        & (race_base["model"] == "logistic_regression")
    ]["oof_macro_f1"].iloc[0]
    rows.append({"setup": "Wyścigi\n2025", "model": "Regresja logistyczna", "macro_f1": race_log})
    for model in ["cnn", "hybrid_cnn_tabular", "gru", "lstm"]:
        val = race_seq[race_seq["model"] == model]["oof_macro_f1"].iloc[0]
        rows.append({"setup": "Wyścigi\n2025", "model": model_label(model), "macro_f1": val})

    plot_df = pd.DataFrame(rows)
    order = ["Regresja logistyczna", "CNN", "Hybryda", "GRU", "LSTM"]
    plt.figure(figsize=(13, 6))
    ax = sns.barplot(
        data=plot_df,
        x="setup",
        y="macro_f1",
        hue="model",
        hue_order=order,
        palette=["#4c78a8", "#59a14f", "#f28e2b", "#b07aa1", "#e15759"],
    )
    ax.set_title("Porównanie modeli w trzech setupach")
    ax.set_xlabel("")
    ax.set_ylabel("Macro F1")
    ax.set_ylim(0, 1)
    ax.legend(title="", ncols=5, loc="upper center", bbox_to_anchor=(0.5, -0.18))
    return savefig("02_model_progression.png")


def plot_training_curve_short_horizon() -> Path:
    history = load_csv("sequence_cnn_training_history")
    plot_df = (
        history.groupby("epoch", as_index=False)
        .agg(
            train_accuracy=("accuracy", "mean"),
            val_accuracy=("val_accuracy", "mean"),
            train_loss=("loss", "mean"),
            val_loss=("val_loss", "mean"),
        )
    )

    fig, axes = plt.subplots(1, 2, figsize=(14, 5.5))
    sns.lineplot(data=plot_df, x="epoch", y="train_accuracy", ax=axes[0], label="trening")
    sns.lineplot(data=plot_df, x="epoch", y="val_accuracy", ax=axes[0], label="walidacja")
    axes[0].set_title("CNN 2023-2025: dokładność")
    axes[0].set_xlabel("Epoka")
    axes[0].set_ylabel("Dokładność")
    axes[0].set_ylim(0, 1)

    sns.lineplot(data=plot_df, x="epoch", y="train_loss", ax=axes[1], label="trening")
    sns.lineplot(data=plot_df, x="epoch", y="val_loss", ax=axes[1], label="walidacja")
    axes[1].set_title("CNN 2023-2025: strata")
    axes[1].set_xlabel("Epoka")
    axes[1].set_ylabel("Strata")
    return savefig("03_cnn_training_curve_short_horizon.png")


def plot_epochs_trained() -> Path:
    rows = []
    for setup, filename in [
        ("Kwalifikacje\n2018-2025", "long_qualifying_top5_sequence_full_rerun_fold_metrics"),
        ("Wyścigi\n2025", "race_2025_top5_sequence_full_rerun_fold_metrics"),
    ]:
        df = load_csv(filename)
        for _, row in df.iterrows():
            rows.append(
                {
                    "setup": setup,
                    "model": model_label(row["model"]),
                    "fold": row["fold"],
                    "epochs_trained": row["epochs_trained"],
                }
            )

    plot_df = pd.DataFrame(rows)
    plt.figure(figsize=(12, 5.5))
    ax = sns.boxplot(data=plot_df, x="model", y="epochs_trained", hue="setup", palette=["#4c78a8", "#f28e2b"])
    sns.stripplot(data=plot_df, x="model", y="epochs_trained", hue="setup", dodge=True, color="black", alpha=0.45, ax=ax)
    handles, labels = ax.get_legend_handles_labels()
    ax.legend(handles[:2], labels[:2], title="")
    ax.set_title("Liczba epok do early stopping")
    ax.set_xlabel("")
    ax.set_ylabel("Epoki")
    return savefig("04_epochs_to_early_stopping.png")


def plot_confusion_matrix() -> Path:
    cm = load_csv("final_model_confusion_matrix")
    matrix = cm.pivot(index="true_driver", columns="pred_driver", values="row_normalized").fillna(0)
    plt.figure(figsize=(7.5, 6))
    ax = sns.heatmap(matrix, annot=True, fmt=".2f", cmap="Blues", vmin=0, vmax=1, square=True)
    ax.set_title("Macierz pomyłek: finalny model interpretowalny")
    ax.set_xlabel("Predykcja")
    ax.set_ylabel("Rzeczywisty kierowca")
    return savefig("05_final_model_confusion_matrix.png")


def plot_feature_importance() -> Path:
    imp = load_csv("final_model_feature_importance_global").head(15).copy()
    imp["feature"] = imp["feature"].str.replace("num__", "", regex=False)
    plt.figure(figsize=(10, 7))
    ax = sns.barplot(data=imp, y="feature", x="mean_abs_coef", color="#4c78a8")
    ax.set_title("Najważniejsze cechy w modelu interpretowalnym")
    ax.set_xlabel("Średnia bezwzględna wartość współczynnika")
    ax.set_ylabel("")
    return savefig("06_driver_feature_importance.png")


def plot_team_bias() -> Path:
    team = load_csv("team_bias_team_classification_summary")
    team_best = (
        team.sort_values(["dataset", "oof_macro_f1"], ascending=[True, False])
        .groupby("dataset", as_index=False)
        .head(1)
    )
    direct = load_csv("hierarchical_team_driver_best_models")
    driver_best = direct[direct["result_name"] == "direct_driver"].copy()

    plot_rows = []
    for _, row in team_best.iterrows():
        plot_rows.append({"dataset": pretty_dataset(row["dataset"]), "task": "Predykcja zespołu", "macro_f1": row["oof_macro_f1"]})
    for _, row in driver_best.iterrows():
        plot_rows.append({"dataset": pretty_dataset(row["dataset"]), "task": "Predykcja kierowcy", "macro_f1": row["macro_f1"]})

    plot_df = pd.DataFrame(plot_rows)
    plt.figure(figsize=(12, 5.8))
    ax = sns.barplot(data=plot_df, x="dataset", y="macro_f1", hue="task", palette=["#f28e2b", "#4c78a8"])
    ax.set_title("Predykcja kierowcy a predykcja zespołu")
    ax.set_xlabel("")
    ax.set_ylabel("Macro F1")
    ax.set_ylim(0, 1)
    ax.legend(title="")
    return savefig("07_driver_vs_team_prediction.png")


def plot_hierarchical_team_aware() -> Path:
    hier = load_csv("hierarchical_team_driver_best_models")
    aware = load_csv("team_aware_driver_model_comparison")
    rows = []
    for _, row in hier[hier["result_name"] == "direct_driver"].iterrows():
        rows.append({"dataset": pretty_dataset(row["dataset"]), "method": "Bezpośrednio: kierowca", "macro_f1": row["macro_f1"]})
    for _, row in hier[hier["result_name"] == "hierarchical_driver_final"].iterrows():
        rows.append({"dataset": pretty_dataset(row["dataset"]), "method": "Najpierw zespół, potem kierowca", "macro_f1": row["macro_f1"]})
    for _, row in aware[aware["model_family"] == "team_aware_tabular"].iterrows():
        rows.append({"dataset": pretty_dataset(row["dataset"]), "method": "Zespół jako cecha pomocnicza", "macro_f1": row["macro_f1"]})

    plot_df = pd.DataFrame(rows)
    plt.figure(figsize=(12, 5.8))
    ax = sns.barplot(data=plot_df, x="dataset", y="macro_f1", hue="method", palette=["#4c78a8", "#e15759", "#59a14f"])
    ax.set_title("Strategie wykorzystania informacji o zespole")
    ax.set_xlabel("")
    ax.set_ylabel("Macro F1")
    ax.set_ylim(0, 1)
    ax.legend(title="", fontsize=11)
    return savefig("08_hierarchical_team_aware_comparison.png")


def plot_same_team_importance() -> Path:
    imp = load_csv("team_bias_same_team_driver_feature_importance")
    sel = imp[
        (imp["task"] == "same_team_ferrari_ham_vs_lec_2025_race")
        & (imp["feature_config"] == "style_core_no_time_context")
        & (imp["class_name"].isin(["HAM", "LEC"]))
        & (imp["rank"] <= 10)
    ].copy()
    sel["feature"] = sel["feature"].str.replace("num__", "", regex=False)
    plt.figure(figsize=(12, 7))
    ax = sns.barplot(data=sel, y="feature", x="coef", hue="class_name", palette=["#4c78a8", "#e15759"])
    ax.axvline(0, color="black", linewidth=1)
    ax.set_title("Ferrari 2025: cechy rozróżniające HAM i LEC")
    ax.set_xlabel("Współczynnik modelu")
    ax.set_ylabel("")
    ax.legend(title="")
    return savefig("09_same_team_ham_lec_feature_importance.png")


def append_mean_history(rows: list[dict], setup: str, filename: str, model: str) -> None:
    history = load_csv(filename)
    if "model" in history.columns:
        history = history[history["model"] == model].copy()
    grouped = (
        history.groupby("epoch", as_index=False)
        .agg(
            train_accuracy=("accuracy", "mean"),
            val_accuracy=("val_accuracy", "mean"),
            train_loss=("loss", "mean"),
            val_loss=("val_loss", "mean"),
        )
    )
    for _, row in grouped.iterrows():
        rows.append({"setup": setup, "model": model_label(model), **row.to_dict()})


def plot_training_curves_multisetup() -> Path:
    rows = []
    append_mean_history(rows, "Kwalifikacje 2023-2025\nHybryda", "sequence_architecture_training_history", "hybrid_cnn_tabular")
    append_mean_history(rows, "Kwalifikacje 2018-2025\nHybryda", "long_qualifying_top5_sequence_full_rerun_training_history", "hybrid_cnn_tabular")
    append_mean_history(rows, "Wyścigi 2025\nHybryda", "race_2025_top5_sequence_full_rerun_training_history", "hybrid_cnn_tabular")
    plot_df = pd.DataFrame(rows)

    fig, axes = plt.subplots(1, 2, figsize=(15, 5.5))
    sns.lineplot(data=plot_df, x="epoch", y="val_accuracy", hue="setup", ax=axes[0], linewidth=2)
    axes[0].set_title("Walidacja: dokładność")
    axes[0].set_xlabel("Epoka")
    axes[0].set_ylabel("Dokładność walidacyjna")
    axes[0].set_ylim(0, 1)
    axes[0].legend(title="", fontsize=10)

    sns.lineplot(data=plot_df, x="epoch", y="val_loss", hue="setup", ax=axes[1], linewidth=2)
    axes[1].set_title("Walidacja: strata")
    axes[1].set_xlabel("Epoka")
    axes[1].set_ylabel("Strata walidacyjna")
    axes[1].legend(title="", fontsize=10)
    return savefig("10_training_curves_multisetup_hybrid.png")


def plot_sequence_fold_variability() -> Path:
    frames = []
    short = load_csv("sequence_architecture_fold_metrics")
    short["setup"] = "Kwalifikacje\n2023-2025"
    frames.append(short[["setup", "model", "fold", "test_macro_f1"]])

    long_df = load_csv("long_qualifying_top5_sequence_full_rerun_fold_metrics")
    long_df["setup"] = "Kwalifikacje\n2018-2025"
    frames.append(long_df[["setup", "model", "fold", "test_macro_f1"]])

    race_df = load_csv("race_2025_top5_sequence_full_rerun_fold_metrics")
    race_df["setup"] = "Wyścigi\n2025"
    frames.append(race_df[["setup", "model", "fold", "test_macro_f1"]])

    plot_df = pd.concat(frames, ignore_index=True)
    plot_df["model"] = plot_df["model"].map(model_label)

    plt.figure(figsize=(13, 6))
    ax = sns.boxplot(data=plot_df, x="model", y="test_macro_f1", hue="setup", palette=["#4c78a8", "#59a14f", "#f28e2b"])
    sns.stripplot(data=plot_df, x="model", y="test_macro_f1", hue="setup", dodge=True, color="black", alpha=0.35, ax=ax)
    handles, labels = ax.get_legend_handles_labels()
    ax.legend(handles[:3], labels[:3], title="")
    ax.set_title("Zmienność wyniku między foldami")
    ax.set_xlabel("")
    ax.set_ylabel("Macro F1 na foldzie testowym")
    ax.set_ylim(0, 1)
    return savefig("11_sequence_fold_variability.png")


def normalized_matrix(cm: pd.DataFrame, model: str) -> pd.DataFrame:
    if "model" in cm.columns:
        cm = cm[cm["model"] == model].copy()
    matrix = cm.pivot(index="true_driver", columns="pred_driver", values="count").fillna(0)
    return matrix.div(matrix.sum(axis=1).replace(0, 1), axis=0)


def plot_best_sequence_confusion_matrices() -> Path:
    items = [
        ("Kwalifikacje 2023-2025\nHybryda", load_csv("sequence_architecture_confusion_matrix"), "hybrid_cnn_tabular"),
        ("Kwalifikacje 2018-2025\nHybryda", load_csv("long_qualifying_top5_sequence_full_rerun_confusion_matrix"), "hybrid_cnn_tabular"),
        ("Wyścigi 2025\nHybryda", load_csv("race_2025_top5_sequence_full_rerun_confusion_matrix"), "hybrid_cnn_tabular"),
    ]
    fig, axes = plt.subplots(1, 3, figsize=(17, 5.2))
    for ax, (title, cm, model) in zip(axes, items):
        matrix = normalized_matrix(cm, model)
        sns.heatmap(matrix, annot=True, fmt=".2f", cmap="Blues", vmin=0, vmax=1, square=True, cbar=ax is axes[-1], ax=ax)
        ax.set_title(title)
        ax.set_xlabel("Predykcja")
        ax.set_ylabel("Rzeczywisty")
    return savefig("12_best_sequence_confusion_matrices.png")


def main() -> None:
    paths = []
    paths.extend(plot_filter_funnels())
    paths.extend([
        plot_model_progression(),
        plot_training_curve_short_horizon(),
        plot_epochs_trained(),
        plot_confusion_matrix(),
        plot_feature_importance(),
        plot_team_bias(),
        plot_hierarchical_team_aware(),
        plot_same_team_importance(),
        plot_training_curves_multisetup(),
        plot_sequence_fold_variability(),
        plot_best_sequence_confusion_matrices(),
    ])
    print("Thesis figure pack created:")
    for path in paths:
        print(f"- {path}")


# Notebook safety: odkomentuj, je?eli chcesz uruchomi? t? sekcj?.
# main()


## Figury narracyjne i wybrane case studies do ko?cowej syntezy

?r?d?o: `14_thesis_narrative_case_study_figures.py`

Je?eli chcesz uruchomi? oryginalny skrypt bez wchodzenia w definicje funkcji, u?yj:

```python
%run 14_thesis_narrative_case_study_figures.py
```


In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd
import seaborn as sns


EXPORT_DIR = Path("exports")
FIG_DIR = Path("figures") / "thesis"
FIG_DIR.mkdir(parents=True, exist_ok=True)


def export_csv(df: pd.DataFrame, name: str) -> Path:
    path = EXPORT_DIR / f"{name}.csv"
    df.to_csv(path, index=False)
    return path


def savefig(name: str) -> Path:
    path = FIG_DIR / name
    plt.tight_layout()
    plt.savefig(path, bbox_inches="tight", dpi=220)
    plt.close()
    return path


def plot_research_arc() -> Path:
    rows = [
        {"stage": "1. Kierowca\nz telemetrii", "result": "Model rozpoznaje kierowcę", "score": 0.9147},
        {"stage": "2. Kontrola\nzespołu", "result": "Zespół też ma podpis", "score": 0.8240},
        {"stage": "3. Ten sam\nzespół", "result": "Styl nie znika w jednym aucie", "score": 0.8820},
        {"stage": "4. Transfer\nstylu", "result": "Część stylu przenosi się", "score": 0.6480},
    ]
    df = pd.DataFrame(rows)
    plt.figure(figsize=(12, 5.5))
    ax = sns.barplot(data=df, x="stage", y="score", color="#4c78a8")
    ax.set_title("Klamra badawcza: od klasyfikacji do interpretacji stylu")
    ax.set_xlabel("")
    ax.set_ylabel("Przykładowa metryka")
    ax.set_ylim(0, 1)
    for idx, row in df.iterrows():
        ax.text(idx, row["score"] + 0.025, row["result"], ha="center", va="bottom", fontsize=10)
    return savefig("22_thesis_research_arc.png")


def plot_transfer_case_studies() -> Path:
    summary = pd.read_csv(EXPORT_DIR / "driver_style_transfer_stability_summary.csv")
    selected = summary[summary["Driver"].isin(["OCO", "GAS", "ALO", "RIC", "SAI", "BOT"])].copy()
    selected["case_type"] = selected["Driver"].map(
        {
            "OCO": "mocny przykład",
            "GAS": "mocny przykład",
            "ALO": "mocny przykład",
            "RIC": "częściowy przykład",
            "SAI": "kontrprzykład",
            "BOT": "kontrprzykład",
        }
    )
    selected = selected.sort_values("median_similarity", ascending=False)
    export_csv(selected, "thesis_selected_transfer_case_studies")

    plt.figure(figsize=(11, 5.8))
    ax = sns.barplot(
        data=selected,
        x="Driver",
        y="median_similarity",
        hue="case_type",
        dodge=False,
        palette={
            "mocny przykład": "#59a14f",
            "częściowy przykład": "#f28e2b",
            "kontrprzykład": "#e15759",
        },
    )
    ax.axhline(0, color="black", linewidth=1)
    ax.set_title("Wybrane przykłady stabilności stylu po zmianie zespołu")
    ax.set_xlabel("Kierowca")
    ax.set_ylabel("Mediana podobieństwa profilu między zespołami")
    ax.set_ylim(-0.05, 0.8)
    ax.legend(title="")
    for container in ax.containers:
        ax.bar_label(container, fmt="%.2f", fontsize=9)
    return savefig("23_selected_driver_transfer_case_studies.png")


def write_narrative_note() -> Path:
    text = """# Proponowana klamra badawcza pracy

## Główna narracja

Praca może być poprowadzona jako przejście od prostego pytania klasyfikacyjnego do interpretacji stylu i dopasowania kierowca-samochód.

1. Najpierw sprawdzam, czy z samej telemetrii można rozpoznać kierowcę.
2. Następnie pokazuję, że wynik nie jest trywialny, bo w danych istnieje także podpis zespołu/samochodu.
3. Potem kontroluję ten problem przez testy w tym samym zespole, np. Ferrari 2025 HAM vs LEC.
4. Na końcu sprawdzam, czy elementy stylu kierowcy są częściowo stabilne po zmianie samochodu.

## Dlaczego to jest mocne

Taka struktura pokazuje, że praca nie kończy się na wyniku klasyfikatora. Model jest użyty jako narzędzie do badania bardziej motorsportowego problemu: czy styl kierowcy jest mierzalny, na ile miesza się z charakterystyką auta i czy może przenosić się między zespołami.

## Jak uniknąć zarzutu cherry-pickingu

W głównej pracy można pokazać wybrane case studies, ale trzeba jawnie napisać kryterium:

- wybieram przykłady z wystarczającą liczbą okrążeń i zmianą zespołu,
- pokazuję zarówno mocne przykłady stabilności, jak i kontrprzykłady,
- pełna tabela wyników znajduje się w dodatku lub w repozytorium.

Rekomendowany zestaw:

- OCO, GAS, ALO jako mocniejsze przykłady stabilności,
- RIC jako częściowo stabilny i bardzo ciekawy motorsportowo przypadek,
- SAI i BOT jako kontrprzykłady, gdzie profil mocniej zależy od samochodu/kontekstu.

## Proponowany wniosek

Wyniki sugerują, że styl kierowcy nie jest ani całkowicie niezależny od samochodu, ani całkowicie przez samochód determinowany. Publiczna telemetria pozwala uchwycić powtarzalne wzorce jazdy, ale ich interpretacja wymaga kontroli wpływu zespołu, samochodu i sezonu.
"""
    path = Path("THESIS_FINAL_RESEARCH_ARC_PL.md")
    path.write_text(text, encoding="utf-8")
    return path


def main() -> None:
    sns.set_theme(style="whitegrid", context="talk")
    plt.rcParams["font.family"] = "DejaVu Sans"
    paths = [
        plot_research_arc(),
        plot_transfer_case_studies(),
    ]
    note = write_narrative_note()
    print("Thesis narrative case-study figures created:")
    for path in paths:
        print(f"- {path}")
    print(f"Note: {note}")


# Notebook safety: odkomentuj, je?eli chcesz uruchomi? t? sekcj?.
# main()
